# RepFR Model Fitting - Kahana2000

Fit multiple CMR variants to the Kahana 2000 serial recall dataset.

**Paradigm:** Serial Recall
**Reference:** @kahana2000interresponse
**Status:** Placeholder - awaiting data preparation and serial recall template

In [ ]:
import os
import papermill as pm

from jaxcmr.helpers import find_project_root

## Dataset Configuration

In [ ]:
# ============================================================================
# DATASET CONFIGURATION - Kahana2000
# ============================================================================
# Paradigm: Serial Recall
# Reference: @kahana2000interresponse
# Notes: Uses fitting_serial.ipynb template (different sim/loss functions)
# Status: Data file not yet prepared
# ============================================================================

DATA_TAG = "Kahana2000"
DATA_PATH = "data/Kahana2000.h5"  # TODO: Verify filename when data is ready
TRIAL_QUERY = "data['list_type'] > 0"  # Adjust based on actual data structure

project_root = find_project_root()
data_file = os.path.join(project_root, DATA_PATH)
template_file = os.path.join(project_root, "templates/fitting_serial.ipynb")

# Check prerequisites
DATA_READY = os.path.exists(data_file)
TEMPLATE_READY = os.path.exists(template_file)

if not DATA_READY:
    print(f"WARNING: Data file not found at {data_file}")
if not TEMPLATE_READY:
    print(f"WARNING: Serial fitting template not found at {template_file}")
    print("Run this after creating templates/fitting_serial.ipynb")

READY = DATA_READY and TEMPLATE_READY
if READY:
    print(f"Ready to fit models.")

In [ ]:
# ============================================================================
# BASE PARAMETERS (shared across all model fits)
# ============================================================================

target_directory = os.path.join(project_root, "projects/repfr/results")
rendered_dir = os.path.join(project_root, "projects/repfr/notebooks/rendered")
templates_dir = os.path.join(project_root, "templates")

os.makedirs(rendered_dir, exist_ok=True)
os.makedirs(os.path.join(target_directory, "fits"), exist_ok=True)
os.makedirs(os.path.join(target_directory, "simulations"), exist_ok=True)
os.makedirs(os.path.join(target_directory, "figures/fitting"), exist_ok=True)

base_params = {
    # Flow control
    "redo_fits": True,
    "redo_sims": True,
    "redo_figures": True,
    "filter_repeated_recalls": True,
    "handle_elis": False,
    
    # Run identification
    "base_run_tag": "serial_fixed_term",  # Different tag for serial recall
    "experiment_count": 200,
    "max_subjects": 0,
    
    # Data configuration
    "base_data_tag": DATA_TAG,
    "data_tag": DATA_TAG,
    "data_path": DATA_PATH,
    "trial_query": TRIAL_QUERY,
    "target_directory": target_directory,
    
    # Component paths (same as free recall for now)
    "component_paths": {
        "mfc_create_fn": "jaxcmr.components.linear_memory.init_mfc",
        "mcf_create_fn": "jaxcmr.components.linear_memory.init_mcf",
        "context_create_fn": "jaxcmr.components.context.init",
        "termination_policy_create_fn": "jaxcmr.components.termination.NoStopTermination",
    },
    
    # Algorithm paths (SERIAL RECALL - TODO: verify these paths exist)
    # These will need to be implemented or adjusted for serial recall
    "sim_alg_path": "jaxcmr.simulation.simulate_study_serial_recall",  # TODO: create this
    "loss_fn_path": "jaxcmr.loss.serial_recall_likelihood.SerialRecallLikelihoodFnGenerator",  # TODO: create this
    "fit_alg_path": "jaxcmr.fitting.ScipyDE",
    
    # Fitting hyperparameters
    "seed": 0,
    "relative_tolerance": 0.001,
    "popsize": 15,
    "num_steps": 1000,
    "cross_rate": 0.9,
    "diff_w": 0.85,
    "best_of": 3,
    
    # Analyses to run after fitting (serial recall specific)
    "comparison_analysis_configs": [
        {"target": "jaxcmr.analyses.srac.plot_srac"},
        {"target": "jaxcmr.analyses.order_error_rate.plot_order_error_rate"},
        {"target": "jaxcmr.analyses.omission_error_rate.plot_omission_error_rate"},
    ],
    "single_analysis_configs": [],
}

## Model Configurations

In [ ]:
# Standard free parameter bounds
STANDARD_FREE_BOUNDS = {
    "encoding_drift_rate": [2.220446049250313e-16, 0.9999999999999998],
    "start_drift_rate": [2.220446049250313e-16, 0.9999999999999998],
    "recall_drift_rate": [2.220446049250313e-16, 0.9999999999999998],
    "shared_support": [2.220446049250313e-16, 100.0],
    "item_support": [2.220446049250313e-16, 100.0],
    "learning_rate": [2.220446049250313e-16, 0.9999999999999998],
    "primacy_scale": [2.220446049250313e-16, 100.0],
    "primacy_decay": [2.220446049250313e-16, 100.0],
    "choice_sensitivity": [2.220446049250313e-16, 100.0],
}

STANDARD_FIXED = {
    "allow_repeated_recalls": False,
    "learn_after_context_update": False,
}

# Model configurations (same models, different paradigm)
varied_parameters = [
    {
        "model_name": "CompositeCMR",
        "make_factory_path": "jaxcmr.models.cmr.make_factory",
        "parameters": {"fixed": STANDARD_FIXED, "free": STANDARD_FREE_BOUNDS},
    },
    {
        "model_name": "InstanceCMR",
        "make_factory_path": "jaxcmr.models.positional_cmr.make_factory",
        "parameters": {"fixed": STANDARD_FIXED, "free": STANDARD_FREE_BOUNDS},
    },
    {
        "model_name": "CompositeCMRNoReinstate",
        "make_factory_path": "jaxcmr.models.no_reinstate_cmr.make_factory",
        "parameters": {"fixed": STANDARD_FIXED, "free": STANDARD_FREE_BOUNDS},
    },
    {
        "model_name": "CompositeCMRDistinct",
        "make_factory_path": "jaxcmr.models.distinct_contexts_cmr.make_factory",
        "parameters": {"fixed": STANDARD_FIXED, "free": STANDARD_FREE_BOUNDS},
    },
    {
        "model_name": "InstanceCMRReinf",
        "make_factory_path": "jaxcmr.models.reinf_positional_cmr.make_factory",
        "parameters": {"fixed": STANDARD_FIXED, "free": STANDARD_FREE_BOUNDS},
    },
]

## Run Model Fitting

In [ ]:
if not READY:
    print("Skipping fitting - prerequisites not met.")
    print("Required:")
    print(f"  - Data: {data_file} {'[OK]' if DATA_READY else '[MISSING]'}")
    print(f"  - Template: {template_file} {'[OK]' if TEMPLATE_READY else '[MISSING]'}")
else:
    for partial_params in varied_parameters:
        params = base_params.copy()
        params.update(partial_params)
        
        data_tag = params["data_tag"]
        model_name = params["model_name"]
        base_run_tag = params["base_run_tag"]
        best_of = params["best_of"]
        max_subjects = params["max_subjects"]
        
        fit_tag = f"{base_run_tag}_best_of_{best_of}"
        if max_subjects:
            fit_tag += f"_nsubs_{max_subjects}"
        
        output_path = os.path.join(
            rendered_dir,
            f"fitting_{data_tag}_{model_name}_{fit_tag}.ipynb"
        )
        
        print(f"Fitting {model_name} -> {output_path}")
        pm.execute_notebook(
            template_file,  # Uses serial template
            output_path,
            parameters=params,
            progress_bar=True,
        )